# Controlled MedGemma 4B Fine-Tuning Pilot for InteractMed-VLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Google-Health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb)

This notebook implements the controlled **Tier 2 Parameter-Efficient Fine-Tuning (PEFT)** pilot for MedGemma 4B on the InteractMed-VLM chest radiograph report generation cohort.

### Research Goals & Scientific Constraints:
- **Model Architecture:** `google/medgemma-4b-it` (Gemma 3 multimodal vision architecture: 4.3B parameters, 2560 embedding dim, fixed SigLIP vision grid).
- **Adaptation Hierarchy (Tier 2):** Low-Rank Adaptation (LoRA) on language decoder attention/MLP and multimodal projector. **Vision encoder is strictly frozen** for scientific baseline attribution.
- **Cohort Governance:** Trained strictly on 200 patient-disjoint development cases (`train_dev`), evaluated on 50 validation cases (`val_dev`).
- **Untouched Test Benchmark:** The 117-case held-out test cohort (`expanded_test`) is permanently locked and unseen.
- **Hardware Target:** NVIDIA A100 GPU (40GB/80GB VRAM) supporting `bfloat16` and 4-bit QLoRA via `bitsandbytes`.

## 1. Environment & Hardware Verification
Ensure the Colab runtime is set to **A100 GPU** (`Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU`).

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU not detected. Please switch to an A100 GPU runtime."
device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"Device: {device_name} (Compute Capability: {capability})")
assert capability[0] >= 8, f"GPU capability {capability} does not support bfloat16. Please select an Ampere/Hopper A100 GPU."

## 2. Install Required Dependencies
MedGemma 4B relies on Gemma 3 multimodal classes introduced in `transformers >= 4.50.0`.

In [ ]:
!pip install --upgrade --quiet "transformers>=4.50.0" peft trl bitsandbytes datasets evaluate accelerate tensorboard

## 3. Hugging Face Authentication
Before downloading `google/medgemma-4b-it`, accept the Health AI Developer Foundations terms of use at https://huggingface.co/google/medgemma-4b-it and authenticate with your HF write token.

In [ ]:
import os, sys
from huggingface_hub import login
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 4. Load Model with 4-Bit QLoRA & AutoProcessor

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_storage=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print("Base model loaded successfully in 4-bit QLoRA precision.")

## 5. Configure LoRA: Freeze Vision Tower & Target Projector + Language Decoder

In [ ]:
from peft import LoraConfig, get_peft_model

# 1. Freeze vision tower strictly for Tier 2 adaptation
frozen_vision_params = 0
for name, param in model.named_parameters():
    if "vision" in name.lower() or "siglip" in name.lower():
        param.requires_grad = False
        frozen_vision_params += param.numel()
print(f"Frozen vision encoder weights: {frozen_vision_params:,} parameters")

# 2. Configure PEFT for language attention/MLP and multimodal projector
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "multi_modal_projector.linear_1", "multi_modal_projector.linear_2"],
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

## 6. Dataset Preparation & Custom Data Collator
Load `train_dev.jsonl` (200 cases) and `val_dev.jsonl` (50 cases). Mask image tokens, special tokens (`boi_token`, `262144`), and prompt instructions so cross-entropy loss is strictly computed over the structured radiologist report tokens (`OBSERVATIONS`, `IMPRESSION`, `LIMITATIONS`).

In [ ]:
import json
from PIL import Image
from typing import Any, List, Dict

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line.strip()) for line in f if line.strip()]

train_data = load_jsonl("artifacts/fine_tuning_cohort/train_dev.jsonl")
val_data = load_jsonl("artifacts/fine_tuning_cohort/val_dev.jsonl")
print(f"Loaded {len(train_data)} train samples and {len(val_data)} validation samples.")

def collate_fn(examples: List[Dict[str, Any]]):
    texts = []
    images = []
    for ex in examples:
        with Image.open(ex["image"]) as img:
            images.append([img.convert("RGB")])
        texts.append(processor.apply_chat_template(
            ex["messages"], add_generation_prompt=False, tokenize=False
        ).strip())
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    
    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100
    if "boi_token" in processor.tokenizer.special_tokens_map:
        boi_id = processor.tokenizer.convert_tokens_to_ids(processor.tokenizer.special_tokens_map["boi_token"])
        labels[labels == boi_id] = -100
    labels[labels == 262144] = -100
    
    # Mask prompt instruction tokens so loss is computed strictly on report output
    for i, ex in enumerate(examples):
        user_text = processor.apply_chat_template([ex["messages"][0]], add_generation_prompt=True, tokenize=False).strip()
        prompt_len = min(len(processor.tokenizer(user_text)["input_ids"]), labels[i].shape[0])
        labels[i, :prompt_len] = -100
    
    batch["labels"] = labels
    return batch

## 7. One-Batch Overfit Sanity Test
Execute 20 optimizer steps on a single batch of 2 cases. Verify that loss drops monotonically before launching full training.

In [ ]:
test_batch = collate_fn(train_data[:2])
optimizer = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=2e-4)
peft_model.train()

print("Running 20-step single-batch overfit test:")
initial_loss = None
for step in range(1, 21):
    optimizer.zero_grad()
    kwargs = {
        "input_ids": test_batch["input_ids"].cuda(),
        "labels": test_batch["labels"].cuda(),
    }
    if "pixel_values" in test_batch and test_batch["pixel_values"] is not None:
        kwargs["pixel_values"] = test_batch["pixel_values"].cuda()
    out = peft_model(**kwargs)
    loss = out.loss
    loss.backward()
    optimizer.step()
    if step == 1:
        initial_loss = loss.item()
    if step % 5 == 0 or step == 1 or step == 20:
        print(f"  Step {step:02d} - Loss: {loss.item():.4f}")

print(f"Initial Loss: {initial_loss:.4f} -> Final Loss: {loss.item():.4f}")
assert loss.item() < initial_loss, "Sanity check failed: loss did not decrease!"
print("PASS: Optimization dynamics verified!")

## 8. SFTTrainer Training Configuration
**Guardrail:** Do not start training automatically without explicit confirmation. Training takes ~20-30 minutes on an NVIDIA A100 GPU.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="checkpoints/medgemma_lora_tier2",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16, # Effective batch size = 16
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=processor,
    data_collator=collate_fn,
)
print("SFTTrainer ready. Run trainer.train() when ready to launch training.")

## 9. Launch Training & Save Checkpoint

In [ ]:
# Uncomment the following lines to execute training when confirmed:
# trainer.train()
# trainer.save_model("checkpoints/medgemma_lora_tier2")
# processor.save_pretrained("checkpoints/medgemma_lora_tier2")
# print("Fine-tuned LoRA checkpoint saved successfully!")

## 10. Audit & Evaluation on Validation Split
Run evaluation comparing Baseline, Prompt v3, and the Fine-Tuned Checkpoint on `val_dev` (N=50).

In [ ]:
!python scripts/evaluate_finetuned_medgemma.py --split val_dev --adapter-path checkpoints/medgemma_lora_tier2